# Capstone · Notebook 3. Inference

*Modern LLMs course · final practical project*

Investigate the trained model in use: time prefill and decode separately, account for KV memory, and check prefix reuse. Then compare approximate weight formats and run speculative decoding. Modules 10 and 11 provide the mechanisms and assumptions against which to read these experiments.

Keep geometric identities, distribution changes, and measured runtime distinct. Quantization is assessed on a broad synthetic vocabulary with several metrics. A harder corpus does not guarantee visible damage, and a low loss does not guarantee its absence.

No network is needed. The notebook uses CPU or an available GPU; runtime depends on the device and budget. Follow the cells in order on a first pass. On a second pass, change one setting and explain which measurement should respond.

## 1. Environment

In [ ]:
import importlib
import sys
import subprocess
import zipfile
from pathlib import Path

ARCHIVE_GLOB = "modern-llms-capstone-*.zip"

def _activate_capstone_root(candidate: Path):
    package_init = candidate / "minillm" / "__init__.py"
    if not package_init.exists():
        return None
    root = str(candidate.resolve())
    if root not in sys.path:
        sys.path.insert(0, root)
    return root

def _find_capstone_root(search_root: Path):
    direct = _activate_capstone_root(search_root)
    if direct:
        return direct
    for child in sorted(search_root.glob("modern-llms-capstone-*")):
        found = _activate_capstone_root(child)
        if found:
            return found
    return None

def _safe_extract(archive_path: Path, destination: Path):
    destination = destination.resolve()
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            member_path = Path(member.filename)
            if member_path.is_absolute() or ".." in member_path.parts:
                raise RuntimeError(f"unsafe path in archive: {member.filename}")
        archive.extractall(destination)

def ensure_minillm():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        found = _find_capstone_root(candidate)
        if found:
            return "project package: " + found

    try:
        import minillm  # noqa: F401
        return "minillm is already importable"
    except ImportError:
        pass

    archives = sorted(Path.cwd().glob(ARCHIVE_GLOB))
    if not archives:
        try:
            from google.colab import files
        except ImportError:
            files = None
        if files is not None:
            print("Upload the Capstone ZIP package from the Stepik lesson. The directory structure will be restored automatically.")
            files.upload()
            archives = sorted(Path.cwd().glob(ARCHIVE_GLOB))

    if archives:
        archive_path = archives[-1]
        destination = Path.cwd() / ".capstone_runtime"
        destination.mkdir(exist_ok=True)
        _safe_extract(archive_path, destination)
        matches = sorted(destination.glob("*/minillm/__init__.py"))
        if len(matches) != 1:
            raise RuntimeError(
                f"expected one minillm package in {archive_path.name}, "
                f"found {len(matches)}"
            )
        root = _activate_capstone_root(matches[0].parent.parent)
        if root:
            return "project archive: " + str(archive_path)

    raise RuntimeError("minillm was not found. Download the language-specific Capstone ZIP package from the Stepik lesson and either extract it in full or upload the ZIP to the current Colab session.")

print(ensure_minillm())

for package in ("torch", "numpy", "matplotlib", "regex"):
    try:
        importlib.import_module(package)
    except ImportError:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", package],
            check=True,
        )

import copy, math, os, random, time
import torch, numpy as np
import matplotlib.pyplot as plt

SEED = 0
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
from minillm.precision import autocast_context, model_parameter_dtype, resolve_compute_dtype
COMPUTE_DTYPE = resolve_compute_dtype(DEVICE, "auto")
print("torch", torch.__version__, "| device:", DEVICE, "| compute:", COMPUTE_DTYPE)

### 1.1. Loading the result of Notebook 2

When the notebooks share a workspace, load `runs/posttrain/final` and generate with the saved policy. If the artifact is absent, the cell reports that fact. Subsequent comparisons use a separately trained model and common inputs, keeping post-training changes out of the quantization experiment.

In [ ]:
from minillm.chat import ChatTemplate, Message
from minillm.generate import SamplingConfig, generate
from minillm.tokenizer import BBPETokenizer
from minillm.train import load_model_artifact

POSTTRAIN_ARTIFACT = Path("runs/posttrain/final")
if (POSTTRAIN_ARTIFACT / "model.pt").exists():
    handoff_model, handoff_meta = load_model_artifact(POSTTRAIN_ARTIFACT, DEVICE)
    handoff_tokenizer = BBPETokenizer.load(POSTTRAIN_ARTIFACT / "tokenizer.json")
    handoff_template = ChatTemplate()
    handoff_ids, _ = handoff_template.encode_with_mask(
        [Message("system", "Answer briefly and directly."),
         Message("user", "What does a tokenizer do?")],
        handoff_tokenizer, add_generation_prompt=True)
    handoff_out, handoff_stats = generate(
        handoff_model, handoff_ids, 40, SamplingConfig(temperature=0.0),
        stop_tokens={handoff_tokenizer.specials["<|eot_id|>"]})
    print("post-training handoff:", handoff_meta)
    print("reply:", repr(handoff_tokenizer.decode(handoff_out[len(handoff_ids):])))
    print("KV bytes:", handoff_stats.kv_bytes)
else:
    print("post-training artifact not found — continuing via the self-contained route")

## 2. A controlled evaluation corpus

Construct a pseudo-language from random words with unequal frequencies. The generating rule is shared while individual sequences differ. This provides controlled comparison inputs; it does not make memorization of a finite training set impossible.

Separate evaluation data before fitting either the tokenizer or the model. Scores on that split describe prediction changes in this synthetic task. Translation, coding, and answer usefulness require their own evaluations.

In [ ]:
from minillm.tokenizer import BBPETokenizer
from minillm.data import TokenStream

def high_entropy_corpus(n_lines: int = 12_000, lexicon: int = 600,
                        seed: int = 0) -> str:
    """A pseudo-language: large lexicon, Zipfian frequencies.

    Unequal frequencies define a controlled prediction task. Loss is measured
    per BPE token, so the entropy of the word distribution is not a direct
    reference value for this metric.
    """
    rng = random.Random(seed)
    alphabet = "abcdefghijklmnopqrstuvwxyz"
    words = ["".join(rng.choice(alphabet) for _ in range(rng.randint(3, 8)))
             for _ in range(lexicon)]
    weights = [1.0 / (i + 1) ** 0.9 for i in range(lexicon)]   # Zipfian tail
    lines = []
    for _ in range(n_lines):
        k = rng.randint(6, 14)
        lines.append(" ".join(rng.choices(words, weights=weights, k=k)) + ".\n")
    return "".join(lines)

corpus = high_entropy_corpus(n_lines=8_000, seed=SEED)
lines = corpus.splitlines(keepends=True)
cut = int(len(lines) * 0.9)
train_text, eval_text = "".join(lines[:cut]), "".join(lines[cut:])
tokenizer = BBPETokenizer.train(train_text, vocab_size=1024, specials=["<|eos|>"])
stream = TokenStream.from_text(train_text, tokenizer, seq_len=128)
eval_stream = TokenStream.from_text(eval_text, tokenizer, seq_len=128)
print(f"corpus {len(corpus):,} characters · vocabulary {tokenizer.vocab_size} · "
      f"{len(stream):,} windows · sha {stream.checksum()}")
print(f"uniform-model reference loss: ln(V) = "
      f"{math.log(tokenizer.vocab_size):.3f}")

## 3. Two models: target and draft

Speculative decoding needs two models sharing a vocabulary: a large one whose
distribution we are obliged to preserve, and a small one that proposes candidates
cheaply. We train both on the same data — that makes the draft resemble the
target, and the acceptance rate depends directly on that resemblance (M11).

In [ ]:
from minillm.config import MiniLLMConfig
from minillm.model import MiniLLM
from minillm.train import TrainConfig, evaluate, train

def build(hidden: int, layers: int, heads: int, kv: int) -> MiniLLM:
    cfg = MiniLLMConfig.profile("S", vocab_size=tokenizer.vocab_size,
                                hidden_size=hidden, num_hidden_layers=layers,
                                num_attention_heads=heads, num_key_value_heads=kv,
                                head_dim=hidden // heads, max_position_embeddings=256)
    return MiniLLM(cfg).to(DEVICE)

# This budget targets CPU execution. Longer training is a separate experiment;
# remeasure both loss and sensitivity to quantization afterwards.
models = {}
for name, (hidden, layers, heads, kv, steps) in {
    "target": (192, 4, 6, 2, 250),
    "draft":  (96, 2, 4, 2, 250),
}.items():
    if os.getenv("CAPSTONE_FAST_VALIDATION") == "1":
        steps = 12
    torch.manual_seed(SEED)
    model = build(hidden, layers, heads, kv)
    print(f"--- {name}: {model.num_parameters():,} parameters")
    train(model, stream, TrainConfig(steps=steps, seq_len=128, micro_batch_size=8,
                                     optimizer="muon", seed=SEED, device=DEVICE, compute_dtype="auto",
                                     log_every=steps // 2))
    models[name] = model

target, draft = models["target"], models["draft"]
eval_cfg = TrainConfig(seq_len=128, micro_batch_size=8, device=DEVICE, compute_dtype="auto")
base_loss = evaluate(target, eval_stream, eval_cfg, batches=(2 if os.getenv("CAPSTONE_FAST_VALIDATION") == "1" else 16))
BASELINE_COMPUTE = resolve_compute_dtype(DEVICE, "auto")
BASELINE_PARAMETER = model_parameter_dtype(target)
BASELINE_LABEL = f"{BASELINE_PARAMETER}/{BASELINE_COMPUTE}"
BASELINE_STORAGE_BITS = 32.0 if BASELINE_PARAMETER == "fp32" else 16.0
print("target eval loss:", base_loss, "uniform reference:", math.log(tokenizer.vocab_size))

## 4. Prefill and decode: what timing can tell us

Prefill processes prompt positions together, while decode reuses earlier keys and values. Weight traffic can limit dense decoding at small batch sizes; longer prefill often makes better use of arithmetic hardware. Module 10 derives the relevant conditions through arithmetic intensity.

Small matrices, Python overhead, and cache operations also affect this implementation. A latency-versus-length curve cannot by itself identify a hardware bottleneck. Record the device, precision, and lengths, then compare measured phases; diagnosing the limiting resource would require profiling.

In [ ]:
from minillm.generate import SamplingConfig, generate

rows = []
for prompt_len in (16, 32, 64, 128):
    prompt = tokenizer.encode(corpus[:4000])[:prompt_len]
    _, stats = generate(target, prompt, 32, SamplingConfig(temperature=0.0))
    rows.append((prompt_len, stats.prefill_seconds * 1000,
                 stats.decode_seconds / stats.generated_tokens * 1000,
                 stats.kv_bytes / 1024))

print(f"{'prompt':>7} {'prefill, ms':>12} {'ms/token decode':>17} {'KV cache, KiB':>15}")
for prompt_len, prefill_ms, decode_ms, kv_kib in rows:
    print(f"{prompt_len:>7} {prefill_ms:>12.2f} {decode_ms:>17.2f} {kv_kib:>15.1f}")

print('Compare measured phases; diagnosing a bottleneck needs repeated measurements and profiling.')

## 5. The KV cache: arithmetic and three attention schemes

The formula from M5: `2 · L · H_kv · d_head · sizeof(dtype)` bytes per token.
It is not quoted here but checked against the memory actually allocated.

In [ ]:
from minillm.config import gqa_variants, kv_bytes_per_token

variants = gqa_variants(target.cfg)
runtime_bytes = 4 if BASELINE_COMPUTE == "fp32" else 2
print(f"{'scheme':>6} {'H_kv':>5} {('bytes/token ' + BASELINE_COMPUTE):>20} {'at 128K context':>20}")
for name, cfg in variants.items():
    per_token = kv_bytes_per_token(cfg, dtype_bytes=runtime_bytes)
    print(f"{name:>6} {cfg.num_key_value_heads:>5} {per_token:>20,} "
          f"{per_token * 131072 / 1024 ** 3:>18.2f} GiB")

# Checking the formula against the actual cache allocation.
prompt = tokenizer.encode(corpus[:2000])[:48]
cache = target.new_cache()
with torch.no_grad():
    with autocast_context(DEVICE, BASELINE_COMPUTE):
        target(torch.tensor([prompt], device=DEVICE), cache=cache)
actual_bytes = cache.nbytes() // (len(prompt) * 2 * target.cfg.num_hidden_layers * target.cfg.kv_dim)
expected = kv_bytes_per_token(target.cfg, dtype_bytes=actual_bytes) * len(prompt)
print(f"\nactual cache dtype: {str(cache.dtype()).replace('torch.', '')}")
print(f"actual {cache.nbytes():,} B == formula {expected:,} B: {cache.nbytes() == expected}")

assert cache.nbytes() == expected

### Prefix caching

Compute a repeated system prefix once and retain its K/V. Clone that cache for each request so one request's suffix cannot enter another's context. The experiment uses greedy decoding with unchanged weights, positions, and masks.

Check token agreement in this run. Logit equivalence is numerical, with a tolerance; rounding can reverse almost tied candidates. Sampling would additionally require aligned randomness. Equal output distributions do not imply identical individual sequences.

In [ ]:
system_prompt = tokenizer.encode(corpus[:1200])[:96]
questions = [tokenizer.encode(corpus[5000 + 500 * i:5000 + 500 * i + 200])[:16]
             for i in range(4)]

warm = target.new_cache()
if DEVICE.startswith("cuda"):
    torch.cuda.synchronize()
t0 = time.perf_counter()
with torch.no_grad():
    with autocast_context(DEVICE, BASELINE_COMPUTE):
        target(torch.tensor([system_prompt], device=DEVICE), cache=warm)
if DEVICE.startswith("cuda"):
    torch.cuda.synchronize()
warmup_ms = (time.perf_counter() - t0) * 1000

cold_total = hot_total = 0.0
identical = True
for tail in questions:
    cold, cold_stats = generate(target, system_prompt + tail, 16,
                                SamplingConfig(temperature=0.0))
    hot, hot_stats = generate(target, system_prompt + tail, 16,
                              SamplingConfig(temperature=0.0), cache=warm.clone())
    identical &= cold == hot
    cold_total += cold_stats.prefill_seconds
    hot_total += hot_stats.prefill_seconds

print(f"results identical: {identical}")
print(f"prefill without cache: {cold_total * 1000:6.1f} ms over {len(questions)} requests")
print(f"prefill with cache:    {hot_total * 1000:6.1f} ms (+{warmup_ms:.1f} ms warmup, once)")
print(f"saving over {len(questions)} requests: "
      f"×{cold_total / max(hot_total, 1e-9):.1f} on the prefill phase")

## 6. Quantization and three kinds of change

Start with storage accounting: a full group of $g$ four-bit weights and an eight-bit scale costs $(4g+8)/g$ bits per weight, or 4.25 at $g=32$ and 4.50 at $g=16$. The implementation report also accounts for partial groups and metadata. Its MXFP4/NVFP4 paths are teaching approximations.

Loss scores target-token probabilities; KL measures distributional change relative to the original model; top-1 agreement counts unchanged most-likely tokens. Their sensitivity depends on the inputs and errors. A top-1 change does not directly measure whether a user notices a difference, so the intended task needs a separate check.

In [ ]:
from minillm.generate import quantize_model

probe = eval_stream.batch(torch.arange(16)).to(DEVICE)
with torch.no_grad():
    with autocast_context(DEVICE, BASELINE_COMPUTE):
        reference_logits, _ = target(probe)
reference_probs = torch.softmax(reference_logits.float(), dim=-1)
reference_top1 = reference_logits.argmax(-1)

print(f"{'format':>7} {'eff bits/wt':>11} {'compr.':>8} {'loss':>8} {'Δloss':>8} "
      f"{'KL, nats':>9} {'top-1 agree':>12}")
print(f"{BASELINE_LABEL:>7} {BASELINE_STORAGE_BITS:>11.2f} {'×1.00':>8} {base_loss:>8.4f} {0.0:>8.4f} "
      f"{0.0:>9.5f} {1.0:>11.1%}")

quant_rows = []
for method in ("int8", "nvfp4", "mxfp4"):
    clone = copy.deepcopy(target)
    report = quantize_model(clone, method)
    loss = evaluate(clone, eval_stream, eval_cfg, batches=(2 if os.getenv("CAPSTONE_FAST_VALIDATION") == "1" else 16))
    with torch.no_grad():
        with autocast_context(DEVICE, BASELINE_COMPUTE):
            logits, _ = clone(probe)
    probs = torch.softmax(logits.float(), dim=-1)
    kl = torch.sum(reference_probs * (torch.log(reference_probs + 1e-12)
                                      - torch.log(probs + 1e-12)), dim=-1).mean().item()
    agreement = (logits.argmax(-1) == reference_top1).float().mean().item()
    quant_rows.append((method, report, loss, kl, agreement))
    print(f"{method:>7} {report['effective_bits_per_weight']:>11.2f} "
          f"{'×' + format(report['compression_vs_runtime_fp32'], '.2f'):>8} {loss:>8.4f} "
          f"{loss - base_loss:>+8.4f} {kl:>9.5f} {agreement:>11.1%}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
names = [BASELINE_LABEL] + [r[0] for r in quant_rows]
colors = ["#4C566A", "#A3BE8C", "#EBCB8B", "#BF616A"]

axes[0].bar(names, [base_loss] + [r[2] for r in quant_rows], color=colors)
axes[0].set_ylim(min([base_loss] + [r[2] for r in quant_rows]) * 0.98, max([base_loss] + [r[2] for r in quant_rows]) * 1.02)
axes[0].set_title("loss (note the clipped axis!)"); axes[0].set_ylabel("nats")

axes[1].bar(names, [0.0] + [r[3] for r in quant_rows], color=colors)
axes[1].set_title("KL(baseline ‖ fake quant)"); axes[1].set_ylabel("nats")

axes[2].bar(names, [1.0] + [r[4] for r in quant_rows], color=colors)
axes[2].set_ylim(0, 1.05); axes[2].set_title("top-1 agreement with baseline")

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

Compare all three plots and notice where the loss axis begins. Similar loss bars can coexist with meaningful distributional change. A format ordering in this run is not a universal ranking: weights, grouping, scales, and the task all matter.

This is fake quantization: rounded weights are returned to a working floating-point format. It studies representation error without implementing packed arithmetic. The timings therefore establish no four-bit inference speedup; that requires a compatible storage format and execution kernels.

## 7. Speculative decoding

The draft proposes `k` tokens in a row and the target verifies them in **one**
pass. The acceptance rule `min(1, p/q)`, with a fallback draw from the normalized
residual `max(0, p−q)`, preserves the target's distribution exactly (M11) —
the speedup is free in quality terms.

The key metric is not the acceptance rate itself but **how many tokens come out
per pass of the large model**. Without speculation it equals one by definition.

In [ ]:
from minillm.generate import speculative_generate

prompt = tokenizer.encode(corpus[:2000])[:32]
print(f"{'k':>3} {'accept':>8} {'tokens/target pass':>21} {'draft passes':>14}")
lookahead_rows = []
for k in (1, 2, 4, 8):
    _, stats = speculative_generate(target, draft, prompt, 48, lookahead=k,
                                    sampling=SamplingConfig(temperature=1.0, seed=SEED))
    lookahead_rows.append((k, stats.acceptance_rate, stats.tokens_per_target_forward,
                           stats.draft_decode_forwards))
    print(f"{k:>3} {stats.acceptance_rate:>7.1%} "
          f"{stats.tokens_per_target_forward:>21.2f} {stats.draft_decode_forwards:>14}")

In [ ]:
# A draft equal to the target: the min(1, p/q) rule must accept everything.
_, twin = speculative_generate(target, copy.deepcopy(target), prompt, 32, lookahead=4,
                               sampling=SamplingConfig(temperature=1.0, seed=SEED))
print(f"draft == target: acceptance {twin.acceptance_rate:.1%}, "
      f"{twin.tokens_per_target_forward:.2f} tokens per pass")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ks = [r[0] for r in lookahead_rows]
axes[0].plot(ks, [r[1] for r in lookahead_rows], "o-", color="#5E81AC")
axes[0].set_xlabel("draft depth k"); axes[0].set_ylabel("acceptance rate")
axes[0].set_title("observed acceptance")
axes[1].plot(ks, [r[2] for r in lookahead_rows], "o-", color="#A3BE8C")
axes[1].axhline(1.0, ls="--", lw=1, color="#4C566A", label="no speculation")
axes[1].set_xlabel("draft depth k")
axes[1].set_ylabel("tokens per target pass")
axes[1].set_title("observed yield per verification"); axes[1].legend()
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

import json
inference_record = {"seed": SEED, "device": DEVICE,
    "compute_dtype": BASELINE_COMPUTE, "parameter_dtype": BASELINE_PARAMETER,
    "evaluation_sha": eval_stream.checksum(), "base_loss": base_loss,
    "quantization": [dict(method=r[0], report=r[1], loss=r[2], kl=r[3], top1=r[4]) for r in quant_rows],
    "speculative": [dict(k=r[0], acceptance=r[1], tokens_per_verification=r[2], draft_decode_passes=r[3]) for r in lookahead_rows],
    "identical_draft_acceptance": twin.acceptance_rate,
    "prefix_greedy_identical": identical}
Path("runs/inference").mkdir(parents=True, exist_ok=True)
Path("runs/inference/measurements.json").write_text(
    json.dumps(inference_record, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")

Acceptance rate and tokens per verification pass describe different aspects of the run. Under a constant acceptance probability, longer drafts increase expected emitted tokens with diminishing returns. Real probabilities can depend on position and history. Neither falling acceptance nor a monotone measured yield curve is guaranteed by the algorithm.

Accepted tokens still cost proposal and verification work. Compare the avoided target passes with both models' costs, keeping prefill separate from verification. Module 11 derives the break-even condition under explicit assumptions. This batch-one experiment does not measure server throughput or its workload dependence.

## 8. Results and their scope

KV size is checked against geometry and actual dtype. Prefix reuse tests previously computed states; quantization compares several kinds of prediction change. Speculation records acceptance and token yield alongside draft work.

The saved report describes particular weights, inputs, and hardware. A small timing sample does not diagnose the bottleneck; fake quantization does not test a packed kernel's speed; pseudo-language does not replace target tasks. Repeat warmed-up timing runs and report variability. Assess quality with held-out items and error analysis.

Notebook 4 connects architecture changes to controlled experiments and builds its final table from the measurements of its own run.